In [ ]:
import nltk
# nltk.download('inaugural')
nltk.download('brown')
nltk.download('punkt_tab')
from nltk.corpus import brown


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
for category in brown.categories():
  txt = brown.words(categories=category)
  print(category, len(txt), txt)


adventure 69342 ['Dan', 'Morgan', 'told', 'himself', 'he', 'would', ...]
belles_lettres 173096 ['Northern', 'liberals', 'are', 'the', 'chief', ...]
editorial 61604 ['Assembly', 'session', 'brought', 'much', 'good', ...]
fiction 68488 ['Thirty-three', 'Scotty', 'did', 'not', 'go', 'back', ...]
government 70117 ['The', 'Office', 'of', 'Business', 'Economics', '(', ...]
hobbies 82345 ['Too', 'often', 'a', 'beginning', 'bodybuilder', ...]
humor 21695 ['It', 'was', 'among', 'these', 'that', 'Hinkle', ...]
learned 181888 ['1', '.', 'Introduction', 'It', 'has', 'recently', ...]
lore 110299 ['In', 'American', 'romance', ',', 'almost', 'nothing', ...]
mystery 57169 ['There', 'were', 'thirty-eight', 'patients', 'on', ...]
news 100554 ['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', ...]
religion 39399 ['As', 'a', 'result', ',', 'although', 'we', 'still', ...]
reviews 40704 ['It', 'is', 'not', 'news', 'that', 'Nathan', ...]
romance 70022 ['They', 'neither', 'liked', 'nor', 'disliked', 'the', 

# Frequency-based Keywords

In [ ]:
import nltk
from nltk.corpus import brown, stopwords
from nltk.tokenize import word_tokenize
from collections import Counter

# Ensure resources are available
nltk.download('brown')
nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
stop_words = set(stopwords.words('english'))

def text_normalization(text):
  tokens = []
  for tok in word_tokenize(text):
    if tok.isalpha() and tok.lower() not in stop_words:
      tokens.append(tok.lower())
  return tokens

def top_frequent(tokens, n=10):
  freqs = Counter(tokens)
  return freqs.most_common(n)

for category in brown.categories():
  txt = " ".join(brown.words(categories=category))
  tokens = text_normalization(txt)
  top_freq = top_frequent(tokens)
  print(category, [w for w, f in top_freq])

adventure ['said', 'would', 'man', 'one', 'could', 'back', 'like', 'time', 'get', 'eyes']
belles_lettres ['one', 'would', 'new', 'man', 'time', 'even', 'could', 'may', 'life', 'world']
editorial ['would', 'one', 'new', 'people', 'united', 'may', 'world', 'american', 'time', 'first']
fiction ['would', 'said', 'could', 'one', 'like', 'man', 'back', 'time', 'came', 'get']
government ['state', 'year', 'states', 'may', 'united', 'new', 'one', 'development', 'government', 'would']
hobbies ['one', 'new', 'may', 'time', 'first', 'many', 'two', 'also', 'good', 'used']
humor ['said', 'one', 'would', 'could', 'time', 'even', 'like', 'way', 'things', 'mother']
learned ['af', 'one', 'may', 'would', 'two', 'first', 'also', 'time', 'must', 'used']
lore ['one', 'would', 'time', 'may', 'new', 'first', 'could', 'people', 'many', 'two']
mystery ['would', 'said', 'could', 'one', 'back', 'like', 'man', 'get', 'two', 'know']
news ['said', 'would', 'new', 'one', 'last', 'two', 'state', 'first', 'year', 'pres

## Are these words informative?
Across genres, the extracted words capture high-frequency lexical material, but they are poor at discriminating genres. Many items recur across almost all categories (e.g. one, would, could, time, new, said), indicating that raw frequency, even after stopword removal, primarily reflects general narrative or expository structure, not genre-specific content.

There are isolated informative signals in some genres (e.g. god, christ in religion; president, state in news), but these are the exception rather than the rule. The dominant pattern is shared vocabulary rather than distinctive lexical markers.

## Which types of words dominate the list?
- Modal verbs and auxiliaries
- General-purpose nouns (e.g., man, time, people, world, year)
- Reporting and narrative verbs (e.g., said, get, came, know, thought)
- Quantifiers and ordinals



# TF-IDF Keywords

In [ ]:
import nltk
from nltk.corpus import brown, stopwords
from nltk.tokenize import word_tokenize
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Ensure required resources
nltk.download('brown')
nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
genres = brown.categories()
stop_words = stopwords.words('english')

In [ ]:
def tfidf_one_vs_rest(target_genre, top_k=10):
    """
    Calculates TF–IDF by contrasting one genre against all others.
    Returns the top_k keywords of the target genre.
    """
    # Documento 1: género-alvo
    doc_target = " ".join(brown.words(categories=target_genre))

    # Documento 2: todos os outros géneros
    other_genres = [g for g in genres if g != target_genre]
    doc_rest = " ".join(brown.words(categories=other_genres))

    documents = [doc_target, doc_rest]

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words=stop_words,
        min_df=1,      # remove hapax
        max_df=0.9     # remove universal terms
    )

    X = vectorizer.fit_transform(documents)
    feature_names = np.array(vectorizer.get_feature_names_out())

    # Linha 0 = género-alvo
    scores = X[0].toarray().ravel()

    top_idx = scores.argsort()[::-1][:top_k]
    return feature_names[top_idx]


In [ ]:
genre_keywords = {}

for g in genres:
    genre_keywords[g] = tfidf_one_vs_rest(g, top_k=10)
# genre_keywords

In [ ]:
for g, kws in genre_keywords.items():
    print(f"{g:15s} {list(kws)}")

adventure       ['jess', 'matsuo', 'ramey', 'brannon', 'tilghman', 'barton', 'langford', 'lalaurie', 'nicolas', 'montero']
belles_lettres  ['hearst', 'gorton', 'steele', 'woodruff', 'miriam', 'trevelyan', 'quiney', 'krim', 'malraux', 'garibaldi']
editorial       ['podger', 'trujillo', 'cannery', 'inquirer', 'autistic', 'megatons', 'cdc', 'connally', 'schiele', 'dekalb']
fiction         ['ada', 'andrei', 'stevie', 'styka', 'juanita', 'jessica', 'hino', 'argiento', 'ludie', 'purdew']
government      ['skywave', 'pursuant', 'sba', 'rupees', 'nighttime', 'subsection', 'unadjusted', 'deduct', '104', 'commuter']
hobbies         ['oersted', 'tappet', 'battens', 'tappets', 'pansies', 'schnabel', 'watercolor', 'acrylic', 'gymnastics', 'mosque']
humor           ['barco', 'letch', 'welch', 'moreland', 'viola', 'crombie', 'blatz', 'workbench', 'gorboduc', 'catatonia']
learned         ['anode', 'gyro', 'staining', 'binomial', 'kohnstamm', 'mg', 'pulmonary', 'zg', 'emission', 'hypothalamic']
lore   

## a.	Which words changed?
* Almost all words changed.
  * Disappeared (typical examples): said, would, one, time, new, could
  * Appeared:
    * Proper names (helva, garibaldi, irenaeus, jubal)
    * Technical terms (anode, binomial, demythologization)
    * Genre-specific vocabulary (subsection in government, gymnastics in hobbies).

## b. Why does TF-IDF down-weight frequent words?
* Because TF-IDF explicitly encodes discriminativeness, not absolute frequency. Due to this property, we modeled one genre against the others.




# Part-of-Speech

In [ ]:
!pip install stanza

In [ ]:
import stanza
from collections import Counter

text = (
    "Time flies like an arrow, but fruit flies like a banana. "
    "I can book a room, or I can book a flight."
)

stanza.download("en")
nlp = stanza.Pipeline(
    lang="en",
    processors="tokenize,pos"
)



INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...
INFO:stanza:File exists: /root/stanza_resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/stanza_resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: en (English):
| Processor | Package         |
-------------------------------
| tokenize  | combined        |
| mwt       | combined        |
| pos       | combined_charlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Done loading processors!


In [ ]:
doc = nlp(text)


In [ ]:
token_pos = []

for sent in doc.sentences:
    for word in sent.words:
        token_pos.append((word.text, word.upos))

token_pos


[('Time', 'NOUN'),
 ('flies', 'VERB'),
 ('like', 'ADP'),
 ('an', 'DET'),
 ('arrow', 'NOUN'),
 (',', 'PUNCT'),
 ('but', 'CCONJ'),
 ('fruit', 'NOUN'),
 ('flies', 'VERB'),
 ('like', 'ADP'),
 ('a', 'DET'),
 ('banana', 'NOUN'),
 ('.', 'PUNCT'),
 ('I', 'PRON'),
 ('can', 'AUX'),
 ('book', 'VERB'),
 ('a', 'DET'),
 ('room', 'NOUN'),
 (',', 'PUNCT'),
 ('or', 'CCONJ'),
 ('I', 'PRON'),
 ('can', 'AUX'),
 ('book', 'VERB'),
 ('a', 'DET'),
 ('flight', 'NOUN'),
 ('.', 'PUNCT')]

## 3a. Which PoS categories are most frequent?

In [ ]:
Counter(token_pos)

Counter({('Time', 'NOUN'): 1,
         ('flies', 'VERB'): 2,
         ('like', 'ADP'): 2,
         ('an', 'DET'): 1,
         ('arrow', 'NOUN'): 1,
         (',', 'PUNCT'): 2,
         ('but', 'CCONJ'): 1,
         ('fruit', 'NOUN'): 1,
         ('a', 'DET'): 3,
         ('banana', 'NOUN'): 1,
         ('.', 'PUNCT'): 2,
         ('I', 'PRON'): 2,
         ('can', 'AUX'): 2,
         ('book', 'VERB'): 2,
         ('room', 'NOUN'): 1,
         ('or', 'CCONJ'): 1,
         ('flight', 'NOUN'): 1})

## 3b.	Which words change PoS depending on context?

In [ ]:
[token for token in token_pos if token[0].lower() in {"flies", "like", "can", "book"}]


[('flies', 'VERB'),
 ('like', 'ADP'),
 ('flies', 'VERB'),
 ('like', 'ADP'),
 ('can', 'AUX'),
 ('book', 'VERB'),
 ('can', 'AUX'),
 ('book', 'VERB')]

## Version spaCy


In [ ]:
import spacy
from collections import Counter

text = (
    "Time flies like an arrow, but fruit flies like a banana. "
    "I can book a room, or I can book a flight."
)


In [ ]:

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)


In [ ]:
[(token.text, token.pos_, token.tag_) for token in doc]


[('Time', 'NOUN', 'NN'),
 ('flies', 'VERB', 'VBZ'),
 ('like', 'ADP', 'IN'),
 ('an', 'DET', 'DT'),
 ('arrow', 'NOUN', 'NN'),
 (',', 'PUNCT', ','),
 ('but', 'CCONJ', 'CC'),
 ('fruit', 'NOUN', 'NN'),
 ('flies', 'NOUN', 'NNS'),
 ('like', 'ADP', 'IN'),
 ('a', 'DET', 'DT'),
 ('banana', 'NOUN', 'NN'),
 ('.', 'PUNCT', '.'),
 ('I', 'PRON', 'PRP'),
 ('can', 'AUX', 'MD'),
 ('book', 'VERB', 'VB'),
 ('a', 'DET', 'DT'),
 ('room', 'NOUN', 'NN'),
 (',', 'PUNCT', ','),
 ('or', 'CCONJ', 'CC'),
 ('I', 'PRON', 'PRP'),
 ('can', 'AUX', 'MD'),
 ('book', 'VERB', 'VB'),
 ('a', 'DET', 'DT'),
 ('flight', 'NOUN', 'NN'),
 ('.', 'PUNCT', '.')]

## 3a. Which PoS categories are most frequent?

In [ ]:
pos_counts = Counter(token.pos_ for token in doc)
pos_counts

Counter({'NOUN': 7,
         'VERB': 3,
         'ADP': 2,
         'DET': 4,
         'PUNCT': 4,
         'CCONJ': 2,
         'PRON': 2,
         'AUX': 2})

## 3b. Which words change PoS depending on context?

In [ ]:
[(token.text, token.pos_) for token in doc if token.text.lower() in {"flies", "like", "can", "book"}]


[('flies', 'VERB'),
 ('like', 'ADP'),
 ('flies', 'NOUN'),
 ('like', 'ADP'),
 ('can', 'AUX'),
 ('book', 'VERB'),
 ('can', 'AUX'),
 ('book', 'VERB')]

# Syntax

In [ ]:
!pip install stanza


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 16.2 MB/s eta 0:00:00


In [ ]:
import stanza

stanza.download("en")

sentences = [
    "The cats chase the mice.",
    "The government approved the new policy."
]


nlp = stanza.Pipeline(
    lang="en",
    processors="tokenize,lemma,pos,depparse"
)

docs = [nlp(sentence) for sentence in sentences]


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...
INFO:stanza:File exists: /root/stanza_resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/stanza_resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |
| depparse  | combined_charlm   |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Done loading processors!


In [ ]:
for doc in docs:
  for sent in doc.sentences:
      for word in sent.words:
          print(
              word.text,
              "\tupos=", word.upos,
              "\thead=", sent.words[word.head - 1].text if word.head > 0 else "ROOT",
              "\tdeprel=", word.deprel
          )


The 	upos= DET 	head= cats 	deprel= det
cats 	upos= NOUN 	head= chase 	deprel= nsubj
chase 	upos= VERB 	head= ROOT 	deprel= root
the 	upos= DET 	head= mice 	deprel= det
mice 	upos= NOUN 	head= chase 	deprel= obj
. 	upos= PUNCT 	head= chase 	deprel= punct
The 	upos= DET 	head= government 	deprel= det
government 	upos= NOUN 	head= approved 	deprel= nsubj
approved 	upos= VERB 	head= ROOT 	deprel= root
the 	upos= DET 	head= policy 	deprel= det
new 	upos= ADJ 	head= policy 	deprel= amod
policy 	upos= NOUN 	head= approved 	deprel= obj
. 	upos= PUNCT 	head= approved 	deprel= punct


In [ ]:
deps = [(w.text, w.deprel) for doc in docs for s in doc.sentences for w in s.words]
Counter(deps)


Counter({('The', 'det'): 2,
         ('cats', 'nsubj'): 1,
         ('chase', 'root'): 1,
         ('the', 'det'): 2,
         ('mice', 'obj'): 1,
         ('.', 'punct'): 2,
         ('government', 'nsubj'): 1,
         ('approved', 'root'): 1,
         ('new', 'amod'): 1,
         ('policy', 'obj'): 1})

## 3.	Which is more explicit about grammatical relations?

* Each token is directly associated with a labeled grammatical function:
  * nsubj --> nominal subject
  * obj --> direct object
  * det --> determiner
  * amod --> adjectival modifier

# OPTIONAL: Keywords and Syntactic Patterns across Genres

In [ ]:
import nltk
from nltk.corpus import brown, stopwords
from collections import Counter, defaultdict
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import math

nltk.download("brown")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
genres = brown.categories()


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
def genre_text(genre):
    return " ".join(
        w.lower()
        for w in brown.words(categories=genre)
        if w.isalpha() and w.lower() not in stop_words
    )

genre_docs = {g: genre_text(g) for g in genres}
documents = list(genre_docs.values())


In [ ]:
def log_likelihood(k, n, K, N):
    # k: freq in target, n: size target
    # K: freq in rest, N: size rest
    E1 = n * (k + K) / (n + N)
    E2 = N * (k + K) / (n + N)
    return 2 * (
        k * math.log((k / E1) if k > 0 else 1) +
        K * math.log((K / E2) if K > 0 else 1)
    )
def llr_keywords(genre, n=10):
    target_words = brown.words(categories=genre)
    rest_words = brown.words(categories=[g for g in genres if g != genre])

    target = Counter(w.lower() for w in target_words if w.isalpha())
    rest = Counter(w.lower() for w in rest_words if w.isalpha())

    target_size = sum(target.values())
    rest_size = sum(rest.values())

    scores = {
        w: log_likelihood(target[w], target_size, rest[w], rest_size)
        for w in target if target[w] >= 5
    }

    return [w for w, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]]


In [ ]:
llr_keywords_by_genre = {g: llr_keywords(g) for g in genres}
llr_keywords_by_genre

{'adventure': ['he',
  'is',
  'his',
  'her',
  'of',
  'him',
  'i',
  'you',
  'had',
  'are'],
 'belles_lettres': ['af',
  'you',
  'hearst',
  'his',
  'of',
  'literature',
  'she',
  'gorton',
  'literary',
  'miriam'],
 'editorial': ['berlin',
  'her',
  'khrushchev',
  'podger',
  'was',
  'editor',
  'has',
  'east',
  'had',
  'nuclear'],
 'fiction': ['he',
  'is',
  'had',
  'was',
  'his',
  'she',
  'of',
  'him',
  'are',
  'her'],
 'government': ['fiscal',
  'he',
  'had',
  'was',
  'states',
  'year',
  'rhode',
  'united',
  'of',
  'development'],
 'hobbies': ['he',
  'your',
  'was',
  'feed',
  'had',
  'her',
  'she',
  'clay',
  'said',
  'hanover'],
 'humor': ['arlene',
  'i',
  'letch',
  'barco',
  'moreland',
  'welch',
  'you',
  'viola',
  'crombie',
  'my'],
 'learned': ['af', 'he', 'you', 'i', 'she', 'his', 'of', 'said', 'had', 'is'],
 'lore': ['wine',
  'i',
  'cattle',
  'palfrey',
  'dartmouth',
  'seeds',
  'ecumenical',
  'negroes',
  'folklore',
  

## Which keywords are clearly genre-specific?
* religion: god, christ, church, spirit, christian, faith, jesus
* lore: folklore, cattle, seeds, wine, ecumenical, negroes
* reviews: music, jazz, musical, audience
* government: fiscal, states, united, year, development



## Which genres have the most distinctive keywords?

Most distinctive:
* religion
* science_fiction
* lore
* reviews
* news

*These genres:*
* contain specialised terminology or named entities,
* show minimal overlap with generic narrative vocabulary,
* are well separated from the rest of the corpus lexically.

Least distinctive
* adventure
* fiction
* mystery
* romance
* learned

*These genres:*
* are dominated by pronouns, auxiliaries, and high-frequency verbs,
* rely more on narrative structure than specialised vocabulary,
* are better distinguished via syntax or discourse patterns than keywords.

# OPTIONAL: 2.	List frequent syntactic patterns, using dependency parsing.

In [127]:
import nltk
import stanza
from collections import Counter, defaultdict

nltk.download("brown")
from nltk.corpus import brown


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!


In [128]:
stanza.download("en")

nlp = stanza.Pipeline(
    lang="en",
    processors="tokenize,lemma,pos,depparse",
    tokenize_pretokenized=False,
    verbose=False
)


In [146]:
def get_verbs(sentence):
    """
    Given a stanza sentence, return all verbs (Word objects).
    """
    return [word for word in sentence.words if word.upos == "VERB"]


In [147]:
def get_direct_dependents(sentence, verb_id):
    """
    Given a stanza sentence and a verb word id,
    return all words that directly depend on this verb.
    """
    dependents = [
        word for word in sentence.words
        if word.head == verb_id
    ]
    return dependents


In [148]:
def extract_verb_dependencies_from_sentence(sentence):
    """
    For a single sentence:
    - identify all verbs
    - for each verb, collect its direct dependents
    Returns a list of (verb, dependents).
    """
    verb_dependency_list = []

    verbs = get_verbs(sentence)

    for verb in verbs:
        dependents = get_direct_dependents(sentence, verb.id)
        dep_info = [(dep.lemma, dep.deprel) for dep in dependents]

        verb_dependency_list.append(
            (verb.lemma, dep_info)
        )

    return verb_dependency_list


In [149]:
def extract_verb_dependencies(text):
    """
    Apply dependency parsing to a text.
    For each sentence, extract verbs and their direct dependents.
    Returns a list (per sentence) of verb–dependent lists.
    """
    doc = nlp(text)
    all_sentences_output = []

    for sentence in doc.sentences:
        sentence_output = extract_verb_dependencies_from_sentence(sentence)
        all_sentences_output.append(sentence_output)

    return all_sentences_output


In [151]:
import nltk
from nltk.corpus import brown

nltk.download("brown")


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!


True

In [152]:
def brown_genre_text(genre, max_sentences=None):
    """
    Return a text string built from Brown sentences of a given genre.
    Optionally limit the number of sentences for speed.
    """
    sents = brown.sents(categories=genre)

    if max_sentences:
        sents = sents[:max_sentences]

    return " ".join(" ".join(sent) for sent in sents)


In [155]:
def extract_verb_dependencies_by_genre(genre, max_sentences=1000):
    """
    For a Brown genre:
    - parse all sentences
    - extract verbs and their direct dependents
    Returns a list of sentence-level verb–dependent lists.
    """
    text = brown_genre_text(genre, max_sentences)
    doc = nlp(text)

    genre_output = []

    for sentence in doc.sentences:
        sent_output = extract_verb_dependencies_from_sentence(sentence)
        if sent_output:  # keep only sentences with verbs
            genre_output.extend(sent_output)

    return genre_output


In [165]:
max_sentences=100
results = {}
for genre in brown.categories():
  result = extract_verb_dependencies_by_genre(genre, max_sentences=max_sentences)
  results[genre] = result



In [166]:
for genre in results:
  lst = [str(v) for v in results[genre]]
  print(f"{genre}: {Counter(lst).most_common(5)}")

adventure: [("('tell', [('Dan', 'nsubj'), ('himself', 'iobj'), ('forget', 'ccomp'), ('.', 'punct')])", 1), ("('forget', [('he', 'nsubj'), ('would', 'aux'), ('Ann', 'obj')])", 1), ("('want', [('he', 'nsubj'), ('certainly', 'advmod'), ('do', 'aux'), ('not', 'advmod'), ('wife', 'obj'), ('.', 'punct')])", 1), ("('marry', [('if', 'mark'), ('he', 'nsubj'), ('have', 'aux'), ('her', 'obj'), (',', 'punct')])", 1), ("('ask', [('marry', 'advcl'), ('he', 'nsubj'), ('would', 'aux'), ('have', 'aux'), ('be', 'aux'), ('trouble', 'obl'), ('.', 'punct')])", 1)]
belles_lettres: [("('lead', [('they', 'nsubj'), ('have', 'aux'), ('also', 'advmod'), ('nation', 'obj'), ('direction', 'obl'), ('.', 'punct')])", 1), ("('rang', [('and', 'cc'), ('objective', 'obl'), ('they', 'nsubj'), ('have', 'aux'), ('have', 'aux'), ('they', 'obl'), ('Southerners', 'obj'), ('.', 'punct')])", 1), ("('call', [('who', 'nsubj:pass'), ('be', 'aux:pass'), ('Bourbon', 'xcomp')])", 1), ("('derive', [('name', 'nsubj'), ('presumably', 'ad

## Which type of verbs dominate each genre?
* Narrative genres (adventure, fiction, mystery, romance):
  * Action and event verbs: go, take, leave, break, marry, ask
  * Mental and intentional verbs: want, decide, know, like, dislike
* News / Editorial / Government
  * Reporting and institutional verbs: say, produce, take, include, propose, provide
  * Procedural or administrative verbs: adjourn, convene, perform
* learned
  * Abstract, observational, and relational verbs: observe, measure, compare, remain, superimpose
* Religion
  * Interpretive and abstract verbs: mean, employ, take, make, be
* Reviews:
  * Performance and evaluation verbs: play, celebrate, ride, melt
Science fiction:
  * Cognitive and experiential verbs: know, grok
  * Process and change verbs: grow, merge, let